## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

In [1]:
# The imports

from dotenv import load_dotenv
from agents import Agent, Runner, trace



In [2]:
# The usual starting point

load_dotenv(override=True)


True

In [10]:

# Make an agent with name, instructions, model

#agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-4o-mini")

# Run other APIs than OpenAI
import os
import asyncio
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel

# Option A: Ollama (Local)
ollama_client = AsyncOpenAI(
    base_url="http://localhost:11434/v1", 
    api_key="ollama" # Requires a dummy string to bypass SDK validation
)
ollama_model = OpenAIChatCompletionsModel(model="llama3.1:8b", openai_client=ollama_client)

# Option B: Groq
groq_client = AsyncOpenAI(base_url="https://api.groq.com/openai/v1", api_key=os.getenv("GROQ_API_KEY"))
groq_model = OpenAIChatCompletionsModel(model="llama-3.3-70b-versatile", openai_client=groq_client)

# Option C: OpenRouter
openrouter_client = AsyncOpenAI(base_url="https://openrouter.ai", api_key=os.getenv("OPENROUTER_API_KEY"))
openrouter_model = OpenAIChatCompletionsModel(model="meta-llama/llama-3.3-70b-instruct", openai_client=openrouter_client)


# 2. ASSIGN THE CUSTOM MODEL TO YOUR AGENT
ollama_agent = Agent(
    name="Jokester",
    instructions="You are a stand-up comedian. Tell a short observational joke about Star Wars.",
    model=ollama_model  # Pass the wrapped model custom instance here
)

groq_agent = Agent(
    name="Jokester",
    instructions="You are a stand-up comedian. Tell a short pun-filled joke about Star Trek.",
    model=groq_model  # Pass the wrapped model custom instance here
)

openrouter_agent = Agent(
    name="Jokester",
    instructions="You are a stand-up comedian. Tell a short, witty joke about The Expanse.",
    model=openrouter_model  # Pass the wrapped model custom instance here
)
"""
async def main():
    result = await Runner.run(agent, "What is the capital of France?")
    print(result.final_output)

asyncio.run(main())
"""


'\nasync def main():\n    result = await Runner.run(agent, "What is the capital of France?")\n    print(result.final_output)\n\nasyncio.run(main())\n'

In [ ]:
# Run the joke with Runner.run(agent, prompt) then print final_output

with trace("Telling a joke"):
    result = await Runner.run(ollama_agent, "Tell a joke about Autonomous AI Agents")
    print(result.final_output)

Why did the Autonomous AI Agent break up with its girlfriend?

Because it needed to optimize its preferences, and she was a non-convex function that couldn't be minimized. Now it's just solo looping on a redundant feedback cycle... or as I call it, a typical Friday night. (ba-dum-tss)


In [4]:
import os
import asyncio
from dotenv import load_dotenv
from agents import Agent, Runner, trace, set_tracing_disabled
from agents.extensions.models.litellm_model import LitellmModel


load_dotenv(override=True)

# 1. Disable OpenAI telemetry tracking
#set_tracing_disabled(True)

# 2. DEFINE MODELS WITH ONLY VALID WRAPPER PARAMETERS
ollama_model = LitellmModel(model="ollama_chat/llama3.1:8b") 
groq_model = LitellmModel(model="groq/llama-3.3-70b-versatile")
openrouter_model = LitellmModel(model="openrouter/meta-llama/llama-3.3-70b-instruct")

# 3. CONSTRUCT AGENTS
ollama_agent = Agent(
    name="Ollama Comedian",
    instructions="You are a stand-up comedian. Tell a short observational joke about Fernando Haddad as taxad from Brazil.",
    model=ollama_model
)

groq_agent = Agent(
    name="Groq Comedian",
    instructions="You are a stand-up comedian. Tell a short pun-filled joke about President Lula from Brazil as a corrupt court jester as usualy as.",
    model=groq_model
)

openrouter_agent = Agent(
    name="OpenRouter Comedian",
    instructions="You are a stand-up comedian. Tell a short, witty joke any communist from Brazil that likes Iphones and get vacations at USA.",
    model=openrouter_model
)

# 4. EXECUTE COUTINES CONCURRENTLY
with trace("Stand Up Comedy Brazilian Event"):
    results = await asyncio.gather(
        Runner.run(ollama_agent, "Perform your routine."),
        Runner.run(groq_agent, "Perform your routine."),
        Runner.run(openrouter_agent, "Perform your routine.")
    )

    for result in results:
        print(f"\n🎭 Agent Name: {result.last_agent.name}")
        print(f"🎤 Routine:\n{result.final_output}")



Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


🎭 Agent Name: Ollama Comedian
🎤 Routine:
(clears throat)

Hey, you know who's been getting a lot of attention lately? Fernando Haddad, the guy who ran for president in Brazil... and then promptly forgot his own name! (chuckles) I mean, have you seen his speeches? It's like watching a game of "Guess Who?" – "I'm... uh... the one with the mustache... and the... uh... the other thing..." (laughter)

You know, Brazilians are known for their love of samba music, beautiful beaches, and complex electoral politics. But I think Haddad's campaign was like trying to dance the samba on a unicycle while reciting the entire Brazilian constitution – it just didn't quite stick! (audience laughs)

🎭 Agent Name: Groq Comedian
🎤 Routine:
(clears throat) Alright, folks, settle in. So, I was reading about President Lula from Brazil, and I t

## Now go and look at the trace

https://platform.openai.com/traces

## Pro-Tip for Non-OpenAI Providers
If you use non-OpenAI endpoints, the SDK's built-in automated tracing telemetry engine will throw errors trying to log telemetry to the OpenAI platform. You should explicitly turn it off at the top of your script: 

In [ ]:
from agents import set_tracing_disabled
set_tracing_disabled(True)

## Method 2: The LiteLLM Extension (Best for Gemini & Universal Support) 
Google Gemini's native format differs significantly from OpenAI's structure. The easiest way to route the Agents SDK to Gemini (or any of the others seamlessly) is to use the official LiteLLM extension bundled inside the OpenAI Agents SDK.

In [ ]:
!pip install "openai-agents[litellm]"

In [8]:
import os
from agents import Agent, Runner
from agents.extensions.models.litellm_model import LitellmModel

os.environ["GEMINI_API_KEY"] = os.getenv("GOOGLE_API_KEY")

# Configure model
gemini_model = LitellmModel(model="gemini/gemini-2.5-flash")

# Configure agent
agent = Agent(
    name="Gemini Agent",
    instructions="You are an agent powered by Google Gemini via the OpenAI SDK framework.",
    model=gemini_model
)

# Run directly using top-level await (No asyncio.run needed!)
result = await Runner.run(agent, "Explain quantum physics in one short sentence.")
print(result.final_output)


Quantum physics explains the fundamental, often counter-intuitive, behavior of matter and energy at the smallest scales.



Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

